# \# <i> RNN for Sentiment Analysis

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("IMDB_Dataset.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

# Text Preprocessing

## # 1> converting to lower case

In [ ]:
df["review"] = df["review"].str.lower().str.strip()

In [ ]:
df.head()

## \# 2> remove the urls


In [ ]:
import re

# sample_text = "abc is the wrod, abc" # abc = xyz
# new_text = re.sub("abc", "xyz", sample_text) # subtitue

# new_text

In [ ]:
def remove_urls(text):
    text = re.sub(r"http\S+","",text) # (pattern, replacement, string) eg - https://www.google.com
    return text
df["review"] = df["review"].apply(remove_urls)

In [ ]:
df.head()

## \# 3> removing html tag

In [ ]:
def remove_html(text):
    text = re.sub(r"<.*?>","",text) 
    return text
df["review"] = df["review"].apply(remove_html)



In [ ]:
df.head()

## \# 4> removing punctuations

In [ ]:
def remove_punctuation(text):
    text = re.sub(r"[^\w\s]", "", text) # (pattern, replacement, string) eg - !, @, ., , etc.
    return text
df["review"] = df["review"].apply(remove_punctuation)

In [ ]:
df.head()

## \# 5> Removing Stopwords

In [ ]:
import nltk # natural language toolkit

nltk.download("punkt") # tokenizer
nltk.download("punkt_tab")
nltk.download("stopwords")


In [ ]:
from nltk.tokenize import word_tokenize # create token
from nltk.corpus import stopwords # english language stowords

In [ ]:
# sample_text = "I like coding in python!"
# tokens = word_tokenize(sample_text)
# tokens

In [ ]:
stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    tokens = word_tokenize(text)

    filtered_tokens = [word for word in tokens if word not in stop_words]

    return " ".join(filtered_tokens)

df["review"] = df["review"].apply(remove_stopwords)



In [ ]:
df.head()

## \# 6> Stemming

In [ ]:
# running -> run
# player -> play

# PorterStemming

In [ ]:
from nltk.stem import PorterStemmer

In [ ]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words =[]

    tokens = word_tokenize(text)
    for token in tokens : 
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [ ]:
df.head()

## \# 7> Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

In [ ]:
df.head()

## \# 8> vectorization (TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [ ]:
y = df["sentiment"]

# \# Dataset and DataLoaders

In [ ]:
from  sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
X_train = X_train.toarray() # sparse matrix to numpy array
X_test = X_test.toarray()

In [ ]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(), # numpy arr to direct tensors
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [ ]:
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=True)

# \# building our RNN

In [ ]:
import torch.nn as nn
import torch.optim as optim

In [ ]:
# Many to One Architecture (Many reviews to one sentiment output (+ve or -ve))
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer                                            
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional (h0 -> shape -> (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x,h0)
        #out -> 1st val =  hidden state of all the timesteps --> (batch, seq len, hidden size)
        #2nd val - final hidden state of last timestep

        out = self.fc(out[:, -1, :]) # out[:,-1,:] -> last timestep
        # probability finally

        return out

In [ ]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

# \# training our RNN

In [ ]:
# unsqueeze() ->add  one additional dimension
# squeeze() -> reduce one dimension

In [ ]:
epochs = 10

for epoch in range(epochs):

    model.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        #2d-> 3d
        xb = xb.unsqueeze(1) # add singleton direction
        outputs = model(xb) # (batch_size ,1 )
        probs = torch.sigmoid(outputs.squeeze()) # (batch_size,) -> probability
        loss = criterion(probs, yb) # compute loss
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        
    print(f"epoch {epoch+1}/{epochs}, loss = {train_loss/len(train_loader)}")  

In [ ]:
# eval

model.eval()

with torch.no_grad():
    
    correct_vals = 0
    total_vals = 0
    
    for xb, yb in test_loader:

        xb = xb.unsqueeze(1)

        outputs = model(xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        total_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print("Accuracy : ", (correct_vals/total_vals)*100)

In [ ]:
# Check sentiment of a new review

def preprocess_text(review):
    review = remove_stopwords(review)
    review = stemming(review)

    # Use the SAME TF-IDF vectorizer used during training
    review = tf.transform([review])

    return review.toarray()


def predict_sentiment(review):

    review = preprocess_text(review)

    # (1, 5000)
    review_tensor = torch.tensor(review, dtype=torch.float32)

    # RNN expects (batch_size, seq_len, input_size)
    review_tensor = review_tensor.unsqueeze(1)

    model.eval()

    with torch.no_grad():

        output = model(review_tensor)

        prob = torch.sigmoid(output).item()

        if prob > 0.5:
            print("Positive Review")
        else:
            print("Negative Review")

        print(f"Probability: {prob:.4f}")

In [ ]:
# New Review
review = input("Enter Review: ")
predict_sentiment(review)

In [ ]:
# New Review
review = input("Enter Review: ")
predict_sentiment(review)

In [ ]:
# New Review
review = input("Enter Review: ")
predict_sentiment(review)